In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np



In [2]:
### Load the trained model, scaler and one-hot encoder pickled files
model=load_model('model.keras')

# Load the scaler and one-hot encoder

with open('scaler.pkl','rb') as file:
    scaler=pickle.load(file)

with open('geo_oh_encoder.pkl', 'rb') as file:
    geo_oh_encoder= pickle.load(file)

with open('label_encoder_Gender.pkl','rb') as file:
    label_encoder_Gender= pickle.load(file)
    

In [3]:
# Sample input data for prediction

input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [4]:
input_df = pd.DataFrame([input_data])

In [5]:
# One-hot encode the 'Geography' column

geo_encoded = geo_oh_encoder.transform(input_df[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=geo_oh_encoder.get_feature_names_out(['Geography'])
)
    


In [6]:
geo_encoded_df
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [7]:
# Encode categorical variables
input_df['Gender']= label_encoder_Gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [8]:
## Concatinate the one-hot encoded 
input_df= pd.concat([input_df.drop("Geography", axis=1), geo_encoded_df], axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [9]:
# Scaling the input data
input_scaler = scaler.transform(input_df)
input_scaler

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [15]:
## Predict the probability of customer churn
prediction= model.predict(input_scaler)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step


array([[0.05758806]], dtype=float32)

In [19]:
prediction_proba= prediction[0][0]
prediction_proba

0.05758806

In [18]:
if prediction_proba >0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is unlikely to churn.")

The customer is unlikely to churn.
